# NACC data importation, analysis and preparation

## Libraries and Data Files
- Import necessary libraries, `pandas` and `numpy`.
- Load several CSV files containing data and metadata for analysis.

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

In [2]:
all_nacc = pd.read_csv('/SeaExpCIFS/NACC_0620/investigator_ftldlbd_nacc65.csv')
nacc_variable = pd.read_csv('../../data/nacc_variable.csv')
nacc_allowable_code = pd.read_csv('../../data/nacc_allowable_code.csv')

## Data Cleaning

- Fill missing values in variable_id column using forward filling method.

In [3]:
nacc_allowable_code['variable_id_conv'] = nacc_allowable_code['variable_id'].fillna(method='ffill')
nacc_allowable_code.rename(columns={"descriptor": "individual_descriptor"}, inplace=True)

## Data Merging

- Merge `nacc_variable` with `nacc_allowable_code` on variable IDs to consolidate variable descriptions and their corresponding codes.
- Filter merged data for specific UDS forms (D1 and D2) to get diagnosis features.

In [4]:
merged_df = pd.merge(nacc_variable, nacc_allowable_code, left_on='id', right_on='variable_id_conv', how='outer')
merged_df_diag = merged_df[(merged_df['form_id'] == 'D1') | (merged_df['form_id'] == 'D2')]

## Data Transformation

- Generate dictionaries for 
    - variable name and text diagnosis
    - descriptor codes and text descriptions for each code.

In [5]:
diag_dict = dict(zip(merged_df_diag['id'], merged_df_diag['descriptor']))
code_dict = {
    id_value: {row['code_1']: row['individual_descriptor'] for index, row in group.iterrows()}
    for id_value, group in merged_df_diag.groupby('id')
}

## Diagnosis Dictionary Creation

- Filter the diagnostic columns for NACC data.
- Iterate through filtered data to map NACC IDs to corresponding diagnostic texts.

In [8]:
from tqdm import tqdm
filtered_nacc = all_nacc[list(diag_dict.keys()) + ["NACCID"]].replace({-4: np.NaN, "-4": np.NaN})
nacc_diag_dict = {}
texts = []
for i, row, in tqdm(filtered_nacc.iterrows()):
    st = ""
    for k, v in dict(row).items():
        if k == "NACCID":
            continue
        try:
            if not pd.isna(v):
                st += f"{diag_dict[k].replace(';', ',')}: {code_dict[k][str(int(v))].replace(';', ',')}; "
        except:
            st += f"{diag_dict[k].replace(';', ',')}: {v}; "
            
    if row['NACCID'] in nacc_diag_dict:
        nacc_diag_dict[row['NACCID']].append(st)
    else:
        nacc_diag_dict[row['NACCID']] = [st]
        
    texts.append(st)

188700it [01:26, 2194.11it/s]


In [12]:
len(texts)

188700

In [13]:
all_nacc["diag_SUMMARY"] = texts

## Data Export

- Save the diagnosis dictionary to a JSON file for further use.

In [14]:
import json

with open('../../data/nacc_diagnosis_text.json', 'w', encoding='utf-8') as fp:
    json.dump(nacc_diag_dict, fp, sort_keys=True, indent=4, ensure_ascii=False)

In [15]:
all_nacc.to_csv("../../data/nacc_with_summary.csv", index=False)